# Gradient descent în profunzime

Optimizarea transformă gradientele în actualizări ale parametrilor. Studiem mai întâi o funcție pătratică, unde geometria este vizibilă, apoi comparăm algoritmii folosiți în modele mari.

## 1. O problemă pătratică

Fie $f(x)=\tfrac12x^TAx-b^Tx$, cu $A$ simetrică pozitiv definită. Gradientul este $\nabla f(x)=Ax-b$, iar minimul satisface $Ax^*=b$. Gradient descent actualizează

$$x_{t+1}=x_t-\eta(Ax_t-b). $$

Pentru convergență avem nevoie de $0<\eta<2/\lambda_{max}(A)$.

In [1]:
import numpy as np
A = np.array([[4., 1.], [1., 3.]])
b = np.array([1., 2.])
x = np.zeros(2)
eta = 0.2
for _ in range(80):
    x -= eta*(A@x-b)
print('soluție iterativă:', np.round(x, 4))
print('soluție exactă:  ', np.round(np.linalg.solve(A,b), 4))

soluție iterativă: [0.0909 0.6364]
soluție exactă:   [0.0909 0.6364]


## 2. Condiționarea controlează viteza

Pe fiecare vector propriu, eroarea este multiplicată cu $1-\eta\lambda_i$. Dacă valorile proprii au scări foarte diferite, un pas sigur pentru direcția abruptă este lent pe direcția plată. Standardizarea, normalizarea și precondiționarea reduc această diferență.

In [2]:
def experiment(eta):
    A = np.diag([1., 4.])
    x = np.array([2., 2.])
    for _ in range(50): x -= eta*(A@x)
    return .5*x@A@x
for eta in [0.02, 0.18, 0.51]:
    value = experiment(eta)
    print(f'eta={eta:.2f} -> ' + ('divergență' if value > 1e3 else f'f={value:.4f}'))

eta=0.02 -> f=0.0015
eta=0.18 -> f=0.0000
eta=0.51 -> divergență


## 3. Momentum

Momentum păstrează o medie exponențială a gradientelor:

$$v_t=\beta v_{t-1}+(1-\beta)g_t,\qquad x_{t+1}=x_t-\eta v_t. $$

Oscilațiile care alternează semnul se anulează, iar direcțiile persistente se acumulează. Efectul este asemănător unei inerții care traversează mai repede văile alungite.

## 4. Adam și corecția biasului

Adam urmărește momentele $m_t$ și $v_t$ ale gradientului. Deoarece pornesc din zero, folosim $\hat m_t=m_t/(1-\beta_1^t)$ și $\hat v_t=v_t/(1-\beta_2^t)$. Actualizarea este

$$\theta_{t+1}=\theta_t-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\varepsilon}. $$

Adam adaptează scara pe coordonate, dar rata de învățare și regularizarea rămân hiperparametri importanți. AdamW separă weight decay de gradientul pierderii.

In [3]:
A = np.diag([1., 30.])
def run(momentum):
    x, v = np.array([3., 1.]), np.zeros(2)
    for _ in range(60):
        g = A@x
        v = momentum*v + g
        x -= 0.03*v
    return np.linalg.norm(x)
print(f'GD distanță:       {run(0.0):.4f}')
print(f'Momentum distanță: {run(0.8):.4f}')

GD distanță:       0.7285
Momentum distanță: 0.0821


## Investigație

1. Reprezintă traiectoriile GD și momentum peste curbele de nivel.
2. Determină experimental intervalul stabil pentru $\eta$ și compară cu teoria.
3. Implementează Adam fără biblioteci și verifică fiecare corecție de bias.
4. Compară gradient clipping by value cu clipping by norm.
5. Explică de ce o metrică zgomotoasă nu demonstrează singură că optimizatorul a eșuat.